In [30]:
import numpy as np
from tqdm import tqdm
import torch
import torchvision
import torchvision.transforms as transforms

from collections import defaultdict

In [31]:
batch_size = 1

random_seed = 1
torch.backends.cudnn.enabled = False
torch.manual_seed(random_seed)

transform = transforms.Compose([
    transforms.ToTensor(),
    lambda x: (x * 255).to(dtype=torch.int32)
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

In [32]:
def reshape_to_tuples(data, dim):
    if isinstance(data, torch.Tensor):
        if data.shape[0] == 1:
            data = data.squeeze().numpy()
    
    rows, cols = data.shape
    row_group_size = rows // dim[0]
    col_group_size = cols // dim[1]
    
    row_indices = []
    col_indices = []
    for i in range(0, row_group_size):
        row_indices.append([j for j in range(i, rows, row_group_size)])
        
    for i in range(0, col_group_size):
        col_indices.append([j for j in range(i, cols, col_group_size)])

    tuples = []
    # indices_matrix = []

    for row_indices_group in row_indices:
        for col_indices_group in col_indices:
            group = []
            indices_tuple = []

            for r_idx in row_indices_group:
                for c_idx in col_indices_group:
                    group.append(data[r_idx, c_idx])
                    # indices_tuple.append((r_idx, c_idx))
            
            # indices_matrix.append(tuple(indices_tuple))
            tuples.append(tuple(group))
            
    return tuples

In [33]:
def freq_pair(tuple_list):
    pairs = defaultdict(int)
    for i in range(len(tuple_list) - 1):
        pair = (tuple_list[i], tuple_list[i+1])
        pairs[pair] += 1
    return pairs

In [34]:
# def freq_pair(tuple_list):
#     pairs = defaultdict(int)
#     for tokens in tuple_list:
#         for i in range(len(tuple_list) - 1):
#             pair = (tokens[i], tokens[i+1])
#             pairs[pair] += 1
#     return pairs

In [35]:
def max_freq_pair(freq_pairs_dic):
    max_freq = None

    for pair, freq in freq_pairs_dic.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    return best_pair, max_freq

In [36]:
def merge(tuple_list, pair, idx):
    new_tuple_list = []
    i = 0
    while i < len(tuple_list):
        if i < len(tuple_list) - 1 and (tuple_list[i], tuple_list[i+1]) == pair:
            new_tuple_list.append(idx)
            i += 2
        else:
            new_tuple_list.append(tuple_list[i])
            i += 1
    return new_tuple_list

In [37]:
# def merge(tuple_list, indices_matrix, pair, idx):
#     new_tuple_list = []
#     new_indices_matrix = []
#     for i in range(len(tuple_list)):
#         tokens_tuple = tuple_list[i]
#         token_indices_tuple = indices_matrix[i]
#         merged_tokens = []
#         merged_indices = []
        
#         j = 0
#         while j < len(tokens_tuple):
#             if j < len(tokens_tuple) - 1 and (tokens_tuple[j], tokens_tuple[j+1]) == pair:
#                 merged_tokens.append(idx)
#                 merged_indices.append(token_indices_tuple[j])
#                 j += 2
#             else:
#                 merged_tokens.append(tokens_tuple[j])
#                 merged_indices.append(token_indices_tuple[j])
#                 j += 1

#         new_tuple_list.append(tuple(merged_tokens))
#         new_indices_matrix.append(tuple(merged_indices))
#     return new_tuple_list, new_indices_matrix

In [38]:
def train(tokens_tuple_list, vocab_size, min_freq=2):
    vocab = defaultdict(str)

    while len(vocab) < vocab_size:
        pair, freq = max_freq_pair(freq_pair(tokens_tuple_list))

        if freq < min_freq:
            break
        
        if pair not in vocab.values():
            idx = str(len(vocab))
            vocab[idx] = pair
        else:
            for key, val in vocab.items():
                if val == pair:
                    idx = key
                    
        tokens_tuple_list = merge(tokens_tuple_list, pair, idx)
    
    return tokens_tuple_list, vocab

In [39]:
dim = (2, 2)

for image, label in tqdm(train_loader):
    ttl = reshape_to_tuples(image, dim)
    new_ttl, vocab = train(ttl, 1000)
    break

  0%|          | 0/60000 [00:00<?, ?it/s]


In [40]:
vocab

defaultdict(str,
            {'0': ((0, 0, 0, 0), (0, 0, 0, 0)),
             '1': ('0', '0'),
             '2': ('1', '1'),
             '3': ((253, 0, 253, 0), (253, 0, 253, 0)),
             '4': ((0, 0, 0, 253), (0, 0, 0, 253)),
             '5': ('2', '0'),
             '6': ('1', '0'),
             '7': ('6', (0, 0, 0, 0)),
             '8': ((0, 253, 0, 253), (0, 253, 0, 253)),
             '9': ((0, 0, 253, 0), (0, 0, 253, 0))})

In [41]:
ttl

[(0, 0, 0, 240),
 (0, 0, 0, 253),
 (0, 0, 0, 253),
 (0, 0, 0, 119),
 (0, 0, 0, 25),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 81, 0),
 (0, 0, 0, 45),
 (0, 0, 0, 186),
 (0, 0, 0, 253),
 (0, 0, 0, 253),
 (0, 0, 0, 150),
 (0, 0, 0, 27),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 16),
 (0, 0, 0, 93),
 (0, 0, 0, 252),
 (0, 0, 0, 253),
 (0, 0, 0, 187),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 249),
 (0, 0, 0, 253),
 (0, 0, 0, 249),
 (0, 0, 0, 64),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 46),
 (0, 0, 0, 130),
 (0, 0, 0, 183),
 (0, 0, 0, 253),
 (0, 0, 0, 253),
 (0, 0, 0, 207),
 (0, 0, 0, 2),
 (0, 0, 0

In [42]:
def reshape_to_tensor(data, tuple_dim, original_dim):
    rows, cols = original_dim
    tuple_rows, tuple_cols = tuple_dim
    row_group_size = rows // tuple_rows
    col_group_size = cols // tuple_cols

    new_data = [torch.tensor([], dtype=torch.uint8)] * rows

    for i in range(tuple_rows):
        new_row = []
        idx = 0
        for r in range(len(data)):
            if r != 0 and r % col_group_size == 0:
                new_data[idx] = torch.cat((new_data[idx], torch.tensor(new_row, dtype=torch.uint8)))
                idx += 1
                new_row = []
            new_row.append(data[r][i])
        new_data[idx] = torch.cat((new_data[idx], torch.tensor(new_row, dtype=torch.uint8)))

    for i in range(tuple_rows, tuple_rows + tuple_cols):
        new_row = []
        idx = row_group_size
        for r in range(len(data)):
            if r != 0 and r % col_group_size == 0:
                new_data[idx] = torch.cat((new_data[idx], torch.tensor(new_row, dtype=torch.uint8)))
                idx += 1
                new_row = []
            new_row.append(data[r][i])
        new_data[idx] = torch.cat((new_data[idx], torch.tensor(new_row, dtype=torch.uint8)))
    
    return new_data

In [43]:
reshape = reshape_to_tensor(ttl, (2, 2), (28, 28))

In [44]:
new_ttl

[(0, 0, 0, 240),
 '4',
 (0, 0, 0, 119),
 (0, 0, 0, 25),
 '2',
 (0, 0, 81, 0),
 (0, 0, 0, 45),
 (0, 0, 0, 186),
 '4',
 (0, 0, 0, 150),
 (0, 0, 0, 27),
 '2',
 (0, 0, 0, 0),
 (0, 0, 0, 16),
 (0, 0, 0, 93),
 (0, 0, 0, 252),
 (0, 0, 0, 253),
 (0, 0, 0, 187),
 '5',
 (0, 0, 0, 0),
 (0, 0, 0, 249),
 (0, 0, 0, 253),
 (0, 0, 0, 249),
 (0, 0, 0, 64),
 '7',
 (0, 0, 0, 46),
 (0, 0, 0, 130),
 (0, 0, 0, 183),
 '4',
 (0, 0, 0, 207),
 (0, 0, 0, 2),
 '7',
 (0, 18, 0, 229),
 (0, 18, 0, 253),
 (0, 126, 0, 253),
 (0, 136, 0, 253),
 (0, 175, 0, 250),
 (0, 26, 0, 182),
 (0, 166, 0, 0),
 (0, 255, 0, 0),
 (0, 247, 0, 0),
 (0, 127, 0, 0),
 '0',
 (3, 0, 39, 0),
 (18, 0, 148, 0),
 '8',
 (0, 253, 0, 253),
 (0, 253, 0, 201),
 (0, 225, 0, 78),
 (0, 172, 0, 0),
 (0, 253, 0, 0),
 (0, 242, 0, 0),
 (30, 195, 0, 0),
 (36, 64, 0, 0),
 (94, 0, 24, 0),
 (154, 0, 114, 0),
 (170, 0, 221, 0),
 (253, 0, 253, 0),
 (0, 253, 0, 253),
 (0, 253, 0, 198),
 (0, 253, 0, 81),
 (0, 251, 0, 2),
 (0, 93, 0, 0),
 (0, 82, 0, 0),
 (0, 82, 0, 

In [ ]:
def dfs(pair, vocab):
    pair = list(pair)
    while isinstance(pair[0], str):
        pair[0] = dfs(vocab[pair[0]], vocab)

    while isinstance(pair[1], str):
        pair[1] = dfs(vocab[pair[1]], vocab)

    if isinstance(pair[0], tuple):
        pair[0] = [pair[0]]
    if isinstance(pair[1], tuple):
        pair[1] = [pair[1]]
    return pair[0] + pair[1] 

In [106]:
decoded = []
for i in new_ttl:
    if isinstance(i, str):
        pair = vocab[i]
        decoded = decoded + dfs(pair, vocab)
    else:
        decoded.append(i)